Week 08: Boosting models

### What is a boosting model?
#### Boosting: Trains different homogenous models sequentially, aiming to correct errors in the previous model. ex XGBoost, AdaBoost, GBM 

In [3]:
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt

from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.model_selection import train_test_split,  GridSearchCV, RepeatedStratifiedKFold
from sklearn.feature_selection import SelectKBest, VarianceThreshold
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

from sklearn.neighbors import  KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error as mse
from sklearn.linear_model import LinearRegression

from sklearn.ensemble import RandomForestRegressor, BaggingRegressor, AdaBoostRegressor, GradientBoostingRegressor, StackingRegressor

from xgboost import XGBRegressor

In [ ]:
ames = pd.read_csv('../data/ames.csv')
ames.head()


,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,Utilities,Lot_Config,...,Fence,Misc_Feature,Misc_Val,Mo_Sold,Year_Sold,Sale_Type,Sale_Condition,Sale_Price,Longitude,Latitude
0,One_Story_1946_and_Newer_All_Styles,Residential_Low_Density,141,31770,Pave,No_Alley_Access,Slightly_Irregular,Lvl,AllPub,Corner,...,No_Fence,NaN,0,5,2010,WD,Normal,215000,-93.619754,42.054035
1,One_Story_1946_and_Newer_All_Styles,Residential_High_Density,80,11622,Pave,No_Alley_Access,Regular,Lvl,AllPub,Inside,...,Minimum_Privacy,NaN,0,6,2010,WD,Normal,105000,-93.619756,42.053014
2,One_Story_1946_and_Newer_All_Styles,Residential_Low_Density,81,14267,Pave,No_Alley_Access,Slightly_Irregular,Lvl,AllPub,Corner,...,No_Fence,Gar2,12500,6,2010,WD,Normal,172000,-93.619387,42.052659
3,One_Story_1946_and_Newer_All_Styles,Residential_Low_Density,93,11160,Pave,No_Alley_Access,Regular,Lvl,AllPub,Corner,...,No_Fence,NaN,0,4,2010,WD,Normal,244000,-93.617320,42.051245
4,Two_Story_1946_and_Newer,Residential_Low_Density,74,13830,Pave,No_Alley_Access,Slightly_Irregular,Lvl,AllPub,Inside,...,Minimum_Privacy,NaN,0,3,2010,WD,Normal,189900,-93.638933,42.060899


In [ ]:

object_cols = ames.select_dtypes(include='object').columns
ames[object_cols] = ames[object_cols].astype('category')

X = ames.drop(columns='Sale_Price')
y = ames['Sale_Price']
X_train, X_test, y_train, y_test = train_test_split(
    X, y,  random_state=42
)

In [ ]:
cat_cols = X_train.select_dtypes(include='category').columns
one_hot_cols = X_train[cat_cols].drop(columns=['Overall_Cond', 'Overall_Qual']).columns.to_list()
ord_cols = ['Overall_Cond', 'Overall_Qual']

ohe = OneHotEncoder(drop = 'first', min_frequency=5, handle_unknown='infrequent_if_exist')  ## Should handle infrequent but still throws warning.

## Note the categories are in order of the columns. 
ord_cats = [
    ['Very_Poor', 'Poor', 'Below_Average', 'Fair', 'Average', 'Above_Average', 'Good', 'Very_Good', 'Excellent'],
    ['Very_Poor', 'Poor', 'Below_Average', 'Fair', 'Average', 'Above_Average', 'Good', 'Very_Good', 'Excellent', 'Very_Excellent']
]

orde = OrdinalEncoder(categories=ord_cats)  # Note that it processes columns and labels in order
col_trans = make_column_transformer((ohe, one_hot_cols), (orde, ord_cols), remainder='passthrough')

In [ ]:
## Fit on training predictors only: `ames` still contains Sale_Price, which remainder='passthrough' would pass through as a feature
ames_preprocessed = col_trans.fit_transform(X_train)

## Improved preprocessing

The professor's notes deliberately keep preprocessing simple. Below we follow his suggestions:

1. Remove outliers (training set only)
2. Remove near-constant (low variance) predictors
3. Ordinal-encode **all** ordered quality/condition columns, not just two
4. Group infrequent categories into an "infrequent"/"Other" level with a higher threshold
5. Encode to suit the model: one hot for sklearn trees, native categoricals for XGBoost
6. Compare with repeated CV (`RepeatedKFold`, since the target is continuous)

In [ ]:
from sklearn.model_selection import RepeatedKFold
from sklearn.base import BaseEstimator, TransformerMixin
import warnings

## Unseen test categories are encoded as zeros by the one hot encoder; that is fine for indicators.
warnings.filterwarnings('ignore', message='Found unknown categories')

#### Outliers

De Cock (who compiled the Ames data) recommends removing houses over 4000 sq ft. Three of the five are partial sales priced far below what their size predicts. We remove them from the **training** set only; the test set is left untouched so the test score stays honest.

In [ ]:
outliers = (X_train['Gr_Liv_Area'] > 4000) & (y_train < 300000)
print(X_train.loc[outliers, ['Gr_Liv_Area', 'Sale_Condition']].assign(Sale_Price=y_train[outliers]))

X_tr = X_train[~outliers]
y_tr = y_train[~outliers]
print(X_train.shape, '->', X_tr.shape)

#### Near-constant predictors

Drop any column where a single value accounts for more than 98% of the training rows. `VarianceThreshold` would need the data encoded and scaled first, so a frequency rule is simpler.

In [ ]:
top_share = X_tr.apply(lambda col: col.value_counts(normalize=True, dropna=False).iloc[0])
low_var_cols = top_share[top_share > 0.98].index.tolist()
top_share[low_var_cols].sort_values(ascending=False)

#### Ordinal columns

All of the quality/condition columns have a natural order. The "No_Basement", "No_Garage", etc. levels go at the bottom.

In [ ]:
qual = ['Poor', 'Fair', 'Typical', 'Good', 'Excellent']
ord_map = {
    'Overall_Qual':   ['Very_Poor', 'Poor', 'Below_Average', 'Fair', 'Average', 'Above_Average', 'Good', 'Very_Good', 'Excellent', 'Very_Excellent'],
    'Overall_Cond':   ['Very_Poor', 'Poor', 'Below_Average', 'Fair', 'Average', 'Above_Average', 'Good', 'Very_Good', 'Excellent'],
    'Exter_Qual':     qual,
    'Exter_Cond':     qual,
    'Heating_QC':     qual,
    'Kitchen_Qual':   qual,
    'Bsmt_Qual':      ['No_Basement'] + qual,
    'Bsmt_Cond':      ['No_Basement'] + qual,
    'Fireplace_Qu':   ['No_Fireplace'] + qual,
    'Garage_Qual':    ['No_Garage'] + qual,
    'Garage_Cond':    ['No_Garage'] + qual,
    'Pool_QC':        ['No_Pool'] + qual,
    'Bsmt_Exposure':  ['No_Basement', 'No', 'Mn', 'Av', 'Gd'],
    'BsmtFin_Type_1': ['No_Basement', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'BsmtFin_Type_2': ['No_Basement', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'Garage_Finish':  ['No_Garage', 'Unf', 'RFn', 'Fin'],
    'Functional':     ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
    'Paved_Drive':    ['Dirt_Gravel', 'Partial_Pavement', 'Paved'],
    'Lot_Shape':      ['Irregular', 'Moderately_Irregular', 'Slightly_Irregular', 'Regular'],
    'Land_Slope':     ['Sev', 'Mod', 'Gtl'],
}

## Check that every level in the data appears in our ordering
for col, order in ord_map.items():
    missing = set(X[col].dropna().unique()) - set(order)
    assert not missing, (col, missing)

ord_map = {col: order for col, order in ord_map.items() if col not in low_var_cols}
print(len(ord_map), 'ordinal columns')

#### One hot version (for sklearn models)

- No `drop='first'`: trees don't need it, and it is what caused the warning in the notes.
- `min_frequency=20` rather than 5. With 10-fold CV a category with 5 rows may not appear in a training fold at all.
- Columns are listed explicitly, so the near-constant columns are dropped (no `remainder='passthrough'`).

In [ ]:
keep_cols = X_tr.columns.drop(low_var_cols)
ord_cols2 = list(ord_map)
nom_cols2 = [c for c in X_tr[keep_cols].select_dtypes(include='category').columns if c not in ord_map]
num_cols2 = X_tr[keep_cols].select_dtypes(include='number').columns.tolist()

ohe2 = OneHotEncoder(min_frequency=20, handle_unknown='infrequent_if_exist', sparse_output=False)
orde2 = OrdinalEncoder(categories=[ord_map[c] for c in ord_cols2],
                       handle_unknown='use_encoded_value', unknown_value=-1)

col_trans_v2 = make_column_transformer((ohe2, nom_cols2), (orde2, ord_cols2), ('passthrough', num_cols2))

print(len(nom_cols2), 'nominal,', len(ord_cols2), 'ordinal,', len(num_cols2), 'numeric')
print('Notes col_trans features:   ', col_trans.fit_transform(X_tr).shape[1])
print('Improved col_trans features:', col_trans_v2.fit_transform(X_tr).shape[1])

#### Native categorical version (for XGBoost)

XGBoost handles categories itself with `enable_categorical=True`, but it does no grouping of rare levels. This small transformer turns ordinal columns into integer codes, groups rare nominal levels into "Other", and drops the near-constant columns. It returns a DataFrame with category dtypes so XGBoost still sees the categoricals. Because it is a transformer, it is refit inside every CV fold.

In [ ]:
class NativeCatPrep(BaseEstimator, TransformerMixin):
    def __init__(self, ord_map, drop_cols=(), min_frequency=20):
        self.ord_map = ord_map
        self.drop_cols = drop_cols
        self.min_frequency = min_frequency

    def _nominal(self, X, col):
        return X[col].astype(object).fillna('None')   ## NaN in Mas_Vnr_Type / Misc_Feature means "none"

    def fit(self, X, y=None):
        X = X.drop(columns=list(self.drop_cols))
        self.nom_cols_ = [c for c in X.select_dtypes(include='category').columns if c not in self.ord_map]
        self.levels_ = {}
        for col in self.nom_cols_:
            counts = self._nominal(X, col).value_counts()
            self.levels_[col] = counts[counts >= self.min_frequency].index.tolist()
        return self

    def transform(self, X):
        X = X.drop(columns=list(self.drop_cols)).copy()
        for col, order in self.ord_map.items():
            X[col] = pd.Categorical(X[col], categories=order, ordered=True).codes   ## unseen -> -1
        for col in self.nom_cols_:
            vals = self._nominal(X, col)
            vals = vals.where(vals.isin(self.levels_[col]), 'Other')
            X[col] = pd.Categorical(vals, categories=self.levels_[col] + ['Other'])
        return X

NativeCatPrep(ord_map, low_var_cols).fit_transform(X_tr).dtypes.astype(str).value_counts()

#### Early stopping with an honest validation set

The notes pass `eval_set=[(X_test, y_test)]`, which picks the number of trees using the test set. Instead, split a validation set off the training data.

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(X_tr, y_tr, test_size=0.2, random_state=42)

native_prep = NativeCatPrep(ord_map, low_var_cols).fit(X_fit)
xgb_es = XGBRegressor(n_estimators=2000,
                      learning_rate=0.05,
                      max_depth=3,
                      enable_categorical=True,
                      max_cat_to_onehot=5,
                      early_stopping_rounds=50)
xgb_es.fit(native_prep.transform(X_fit), y_fit,
           eval_set=[(native_prep.transform(X_val), y_val)], verbose=False)

n_trees = xgb_es.best_iteration + 1
print('Best number of trees:', n_trees)

#### Compare encodings with repeated CV

Same XGBoost settings for every model, so only the preprocessing differs. All models are cross-validated on the same (outlier-removed) training rows. Scores are RMSE in dollars, lower is better.

In [ ]:
xgb_params = dict(n_estimators=n_trees, learning_rate=0.05, max_depth=3, random_state=42)

models = {
    'onehot_notes': make_pipeline(col_trans, XGBRegressor(**xgb_params)),
    'onehot_v2':    make_pipeline(col_trans_v2, XGBRegressor(**xgb_params)),
    'native_raw':   XGBRegressor(**xgb_params, enable_categorical=True, max_cat_to_onehot=5),
    'native_v2':    make_pipeline(NativeCatPrep(ord_map, low_var_cols),
                                  XGBRegressor(**xgb_params, enable_categorical=True, max_cat_to_onehot=5)),
}

cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=1)
results = {}
for name, model in models.items():
    results[name] = -cross_val_score(model, X_tr, y_tr, cv=cv,
                                     scoring='neg_root_mean_squared_error', n_jobs=-1)
    print('>%-13s RMSE %.0f (%.0f)' % (name, results[name].mean(), results[name].std()))

plt.boxplot(list(results.values()), tick_labels=list(results.keys()), showmeans=True)
plt.ylabel('CV RMSE ($)')
plt.show()

#### Test set

Fit each model on the training set and score once on the untouched test set. The last row shows the effect of outlier removal by fitting `native_v2` on the training set with the outliers still in.

In [ ]:
rows = []
for name, model in models.items():
    model.fit(X_tr, y_tr)
    pred = model.predict(X_test)
    rows.append((name, model.score(X_test, y_test), np.sqrt(mse(y_test, pred))))

model = models['native_v2'].fit(X_train, y_train)
rows.append(('native_v2 (with outliers)', model.score(X_test, y_test), np.sqrt(mse(y_test, model.predict(X_test)))))

pd.DataFrame(rows, columns=['model', 'test R2', 'test RMSE']).round(3)

#### Why does removing outliers hurt the test score?

The test set contains one of the five big houses: a 4676 sq ft partial sale for $184,750. A model that never saw such sales predicts its full size-based value, and that single squared error drags test R2 down. Excluding that one house, both models score about the same. Removing the outliers makes the model better for ordinary sales, but worse at predicting this rare kind of sale.

In [ ]:
big = (X_test['Gr_Liv_Area'] > 4000).values
for label, X_fit_on, y_fit_on in [('without outliers', X_tr, y_tr), ('with outliers', X_train, y_train)]:
    pred = models['native_v2'].fit(X_fit_on, y_fit_on).predict(X_test)
    r2_rest = 1 - ((y_test[~big] - pred[~big])**2).sum() / ((y_test[~big] - y_test[~big].mean())**2).sum()
    print('%-17s big house: actual %d, predicted %.0f | test R2 without it: %.3f'
          % (label, y_test[big].iloc[0], pred[big][0], r2_rest))